
# End-to-end Purview lineage: Source → Azure ML Dataset → Training Process → ML Model

This notebook demonstrates **full lineage** in **Microsoft Purview** for an Azure ML workflow: a **source dataset** (e.g., ADLS/SQL) → an **Azure ML Dataset** → a **training process** → an **ML Model**. We use Purview's **Apache Atlas** REST APIs to create assets and lineage relationships, authenticated with **Microsoft Entra ID** (`DefaultAzureCredential`).

> ℹ️ Background: Purview supports lineage via Atlas Data Map APIs (entities, relationships, lineage queries). You can build **custom lineage** between DataSet and Process types, and query lineage by GUID. See Microsoft documentation on lineage APIs and custom lineage.



## 0. Install packages
Run once per environment. In Azure ML, `%pip` is supported.


In [ ]:

%pip install -q azure-identity requests pandas
# Optional: if you prefer SDK helpers for Atlas, uncomment:
# %pip install -q pyapacheatlas



## 1. Configuration & credentials
Populate placeholders or set environment variables before running. The notebook uses `DefaultAzureCredential` (Managed Identity in Azure ML, or `az login` locally).


In [ ]:

import os, json, time
from typing import Optional, Dict
import pandas as pd
import requests
from azure.identity import DefaultAzureCredential

# Purview configuration
PURVIEW_ACCOUNT = os.getenv('PURVIEW_ACCOUNT', '<your-purview-account-name>')  # e.g., contoso-purview
PURVIEW_ENDPOINT = f'https://{PURVIEW_ACCOUNT}.purview.azure.com'
DATAMAP_BASE = f'{PURVIEW_ENDPOINT}/datamap/api/atlas/v2'
SCOPE = 'https://purview.azure.net/.default'

# Azure ML identifiers (used to build qualifiedNames)
SUBSCRIPTION_ID = os.getenv('SUBSCRIPTION_ID', '<sub-id>')
RESOURCE_GROUP = os.getenv('RESOURCE_GROUP', '<rg-name>')
WORKSPACE_NAME = os.getenv('WORKSPACE_NAME', '<aml-workspace>')

# Example asset identifiers
SOURCE_KIND = os.getenv('SOURCE_KIND', 'adls')  # adls|sql
# For ADLS:
STORAGE_ACCOUNT = os.getenv('STORAGE_ACCOUNT', '<storageacct>')
SOURCE_CONTAINER = os.getenv('SOURCE_CONTAINER', 'raw')
SOURCE_PATH = os.getenv('SOURCE_PATH', 'data/credit/credit.csv')
# For SQL (if you switch SOURCE_KIND='sql'):
SQL_SERVER = os.getenv('SQL_SERVER', '<server>.database.windows.net')
SQL_DB = os.getenv('SQL_DB', '<db>')
SQL_SCHEMA = os.getenv('SQL_SCHEMA', 'dbo')
SQL_TABLE = os.getenv('SQL_TABLE', 'credit')

# AML dataset & model identifiers
AML_DATASET_NAME = os.getenv('AML_DATASET_NAME', 'credit_dataset')
AML_DATASET_VERSION = os.getenv('AML_DATASET_VERSION', '1')
TRAIN_JOB_ID = os.getenv('TRAIN_JOB_ID', 'train-0001')
MODEL_NAME = os.getenv('MODEL_NAME', 'credit-risk-model')
MODEL_VERSION = os.getenv('MODEL_VERSION', '1')

credential = DefaultAzureCredential(exclude_interactive_browser_credential=False)
print('Config loaded. Update placeholders via environment variables or above constants.')



## 2. Purview helper functions (Atlas Data Map REST)
We implement minimal helpers for:
- **Get/preview token**
- **Create or update entity** (by typeName + qualifiedName)
- **Get entity by unique attribute**
- **Create lineage relationships** using built-in relationship types:
  - `dataset_process_inputs`
  - `process_dataset_outputs`
- **Query lineage** for verification


In [ ]:

def _auth_header():
    token = credential.get_token(SCOPE).token
    return {
        'Authorization': f'Bearer {token}',
        'Content-Type': 'application/json'
    }

def get_token_preview():
    t = credential.get_token(SCOPE).token
    return t[:16] + '...' + t[-16:]

def get_entity_by_unique_attribute(type_name: str, qualified_name: str) -> Optional[Dict]:
    # GET {endpoint}/entity/uniqueAttribute/type/{typeName}?attr:qualifiedName={qn}
    qn = requests.utils.quote(qualified_name, safe='')
    url = f"{DATAMAP_BASE}/entity/uniqueAttribute/type/{type_name}?attr:qualifiedName={qn}"
    resp = requests.get(url, headers=_auth_header(), timeout=60)
    if resp.status_code == 200:
        return resp.json()
    return None

def create_or_update_entity(type_name: str, qualified_name: str, name: str, attributes: Optional[Dict] = None) -> Dict:
    payload = {
        'entity': {
            'typeName': type_name,
            'attributes': {
                'qualifiedName': qualified_name,
                'name': name
            }
        }
    }
    if attributes:
        payload['entity']['attributes'].update(attributes)
    url = f"{DATAMAP_BASE}/entity"
    resp = requests.post(url, headers=_auth_header(), data=json.dumps(payload), timeout=60)
    resp.raise_for_status()
    return resp.json()

def create_relationship(rel_type: str, end1_guid: str, end1_type: str, end2_guid: str, end2_type: str) -> Dict:
    payload = {
        'typeName': rel_type,
        'end1': {'guid': end1_guid, 'typeName': end1_type},
        'end2': {'guid': end2_guid, 'typeName': end2_type}
    }
    url = f"{DATAMAP_BASE}/relationship"
    resp = requests.post(url, headers=_auth_header(), data=json.dumps(payload), timeout=60)
    resp.raise_for_status()
    return resp.json()

def get_lineage(guid: str, direction: str = 'BOTH', depth: int = 2) -> Dict:
    url = f"{DATAMAP_BASE}/lineage/{guid}?direction={direction}&depth={depth}"
    resp = requests.get(url, headers=_auth_header(), timeout=60)
    resp.raise_for_status()
    return resp.json()

print('Token preview:', get_token_preview())



## 3. Define qualifiedNames for assets
Qualified names should be globally unique and stable. We construct readable qualifiedNames for each asset.


In [ ]:

def qn_source():
    if SOURCE_KIND == 'adls':
        return f"https://{STORAGE_ACCOUNT}.dfs.core.windows.net/{SOURCE_CONTAINER}/{SOURCE_PATH}"
    else:
        return f"sql://{SQL_SERVER}/{SQL_DB}/{SQL_SCHEMA}/{SQL_TABLE}"

def qn_aml_dataset():
    return f"azureml://subscriptions/{SUBSCRIPTION_ID}/resourcegroups/{RESOURCE_GROUP}/workspaces/{WORKSPACE_NAME}/datasets/{AML_DATASET_NAME}/{AML_DATASET_VERSION}"

def qn_prep_process():
    return f"azureml://subscriptions/{SUBSCRIPTION_ID}/resourcegroups/{RESOURCE_GROUP}/workspaces/{WORKSPACE_NAME}/jobs/{TRAIN_JOB_ID}-prep"

def qn_train_process():
    return f"azureml://subscriptions/{SUBSCRIPTION_ID}/resourcegroups/{RESOURCE_GROUP}/workspaces/{WORKSPACE_NAME}/jobs/{TRAIN_JOB_ID}"

def qn_model():
    return f"azureml://subscriptions/{SUBSCRIPTION_ID}/resourcegroups/{RESOURCE_GROUP}/workspaces/{WORKSPACE_NAME}/models/{MODEL_NAME}/{MODEL_VERSION}"

print('Example qualified names:')
print('Source      ', qn_source())
print('AML Dataset ', qn_aml_dataset())
print('Prep Proc   ', qn_prep_process())
print('Train Proc  ', qn_train_process())
print('Model       ', qn_model())



## 4. Create (or reuse) assets in Purview
We use generic Atlas types:
- **DataSet** for source, AML dataset, and model
- **Process** for preparation and training steps

> If your AML workspace is registered in Purview (preview), real AML assets (Models/Datasets/Jobs) may already exist and be auto-pushed daily. You can swap the creation below with `get_entity_by_unique_attribute` to pull their GUIDs and then only create relationships.


In [ ]:

# Create or reuse DataSet: Source
src_qn = qn_source()
src_entity = get_entity_by_unique_attribute('DataSet', src_qn)
if not src_entity:
    src_entity = create_or_update_entity('DataSet', src_qn, name='source_dataset')
src_guid = src_entity['entity']['guid'] if 'entity' in src_entity else src_entity.get('guid')
print('Source GUID:', src_guid)

# Create or reuse DataSet: AML dataset
ds_qn = qn_aml_dataset()
ds_entity = get_entity_by_unique_attribute('DataSet', ds_qn)
if not ds_entity:
    ds_entity = create_or_update_entity('DataSet', ds_qn, name=AML_DATASET_NAME)
ds_guid = ds_entity['entity']['guid'] if 'entity' in ds_entity else ds_entity.get('guid')
print('AML Dataset GUID:', ds_guid)

# Create or reuse Process: preparation
prep_qn = qn_prep_process()
prep_entity = get_entity_by_unique_attribute('Process', prep_qn)
if not prep_entity:
    prep_entity = create_or_update_entity('Process', prep_qn, name='ml_data_prep')
prep_guid = prep_entity['entity']['guid'] if 'entity' in prep_entity else prep_entity.get('guid')
print('Prep Process GUID:', prep_guid)

# Create or reuse Process: training
train_qn = qn_train_process()
train_entity = get_entity_by_unique_attribute('Process', train_qn)
if not train_entity:
    train_entity = create_or_update_entity('Process', train_qn, name='ml_training')
train_guid = train_entity['entity']['guid'] if 'entity' in train_entity else train_entity.get('guid')
print('Train Process GUID:', train_guid)

# Create or reuse DataSet: model
model_qn = qn_model()
model_entity = get_entity_by_unique_attribute('DataSet', model_qn)
if not model_entity:
    model_entity = create_or_update_entity('DataSet', model_qn, name=MODEL_NAME)
model_guid = model_entity['entity']['guid'] if 'entity' in model_entity else model_entity.get('guid')
print('Model GUID:', model_guid)



## 5. Build lineage relationships
We create lineage edges using built-in relationship types:
- **`dataset_process_inputs`**: DataSet → Process (input)
- **`process_dataset_outputs`**: Process → DataSet (output)

Flow we build:
1. **Source DataSet** → *(dataset_process_inputs)* → **Prep Process** → *(process_dataset_outputs)* → **AML DataSet**
2. **AML DataSet** → *(dataset_process_inputs)* → **Training Process** → *(process_dataset_outputs)* → **Model DataSet**


In [ ]:

try:
    r1 = create_relationship('dataset_process_inputs', src_guid, 'DataSet', prep_guid, 'Process')
    r2 = create_relationship('process_dataset_outputs', prep_guid, 'Process', ds_guid, 'DataSet')
    r3 = create_relationship('dataset_process_inputs', ds_guid, 'DataSet', train_guid, 'Process')
    r4 = create_relationship('process_dataset_outputs', train_guid, 'Process', model_guid, 'DataSet')
    print('Relationships created:')
    print('r1:', r1.get('guid', 'ok'))
    print('r2:', r2.get('guid', 'ok'))
    print('r3:', r3.get('guid', 'ok'))
    print('r4:', r4.get('guid', 'ok'))
except Exception as e:
    print('Error creating relationships:', e)



## 6. Verify lineage for the model
Query lineage from the **model** GUID (direction `BOTH`, depth `2`).


In [ ]:

try:
    lineage = get_lineage(model_guid, direction='BOTH', depth=2)
    print(json.dumps(lineage, indent=2)[:2000])
except Exception as e:
    print('Lineage query error:', e)



## 7. (Optional) Using existing AML assets auto-pushed to Purview (Preview)
If your **Azure ML workspace is registered in Purview** (preview feature), Purview can automatically receive metadata for **Models, Datasets, and Jobs** daily. Replace the creation in Section 4 with **lookups** to those assets and then only create relationships.

Example lookup by type + qualifiedName:
```python
aml_model_qn = qn_model()  # same format as above
existing = get_entity_by_unique_attribute('<aml-model-type>', aml_model_qn)
model_guid = existing['entity']['guid']
```
> Note: Type names for AML assets vary with Purview's AML integration. Inspect your catalog to find the typeName (e.g., via search or get-by-unique-attributes).



## 8. RBAC & troubleshooting
- Ensure your identity has **Data Curator/Data Reader** roles in Purview to create entities/relationships.
- Private endpoints/DNS: the compute must resolve and reach the Purview endpoint.
- Use correct OAuth scope: `https://purview.azure.net/.default`.
- Relationship types used here (`dataset_process_inputs`, `process_dataset_outputs`) are built-in lineage relationships for **DataSet ↔ Process**.
- Idempotency: we use `Get By Unique Attribute` (type + qualifiedName) to avoid duplicates.
